In [4]:
import pandas as pd

golden_annotation = pd.read_csv(
    "../evaluation/golden_set_annotation.csv"
)

print("Golden Set:", len(golden_annotation))

Golden Set: 200


In [5]:
from sklearn.model_selection import train_test_split

X = golden_annotation["customer_message"]
y = golden_annotation["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training:", len(X_train))
print("Test:", len(X_test))

Training: 150
Test: 50


In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

majority_baseline = DummyClassifier(strategy="most_frequent")

majority_baseline.fit(X_train, y_train)

majority_predictions = majority_baseline.predict(X_test)

majority_accuracy = accuracy_score(y_test, majority_predictions)
majority_macro_f1 = f1_score(
    y_test,
    majority_predictions,
    average="macro"
)
majority_weighted_f1 = f1_score(
    y_test,
    majority_predictions,
    average="weighted"
)

print("Majority Class Baseline")
print("Accuracy:", round(majority_accuracy, 3))
print("Macro F1:", round(majority_macro_f1, 3))
print("Weighted F1:", round(majority_weighted_f1, 3))
print(
    "Predicted class:",
    majority_baseline.classes_[majority_baseline.class_prior_.argmax()]
)

Majority Class Baseline
Accuracy: 0.26
Macro F1: 0.028
Weighted F1: 0.107
Predicted class: non_support


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=30000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

baseline_lr.fit(X_train, y_train)

print("Baseline 1 trained successfully")

Baseline 1 trained successfully


In [11]:
from sklearn.metrics import classification_report, f1_score

y_pred_lr = baseline_lr.predict(X_test)

print("Macro F1:", f1_score(y_test, y_pred_lr, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_lr, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, zero_division=0))

Macro F1: 0.19807017543859648
Weighted F1: 0.2914736842105263

Classification Report:
                     precision    recall  f1-score   support

 airport_gate_staff       0.00      0.00      0.00         2
            baggage       0.00      0.00      0.00         3
     booking_change       0.17      0.50      0.25         2
           check_in       0.00      0.00      0.00         1
    contact_support       0.00      0.00      0.00         2
       fees_charges       0.00      0.00      0.00         3
  flight_disruption       0.29      0.40      0.33         5
 flight_information       0.00      0.00      0.00         2
          follow_up       0.50      0.50      0.50         8
    loyalty_upgrade       0.00      0.00      0.00         2
        non_support       0.67      0.31      0.42        13
   onboard_aircraft       0.00      0.00      0.00         1
      other_unclear       0.00      0.00      0.00         2
refund_compensation       0.67      1.00      0.80         

In [12]:
from sklearn.svm import LinearSVC

baseline_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=30000
    )),
    ("classifier", LinearSVC(
        class_weight="balanced"
    ))
])

baseline_svm.fit(X_train, y_train)

y_pred_svm = baseline_svm.predict(X_test)

print("Macro F1:", f1_score(y_test, y_pred_svm, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred_svm, average="weighted"))

Macro F1: 0.26
Weighted F1: 0.32899999999999996


In [13]:
baseline_results = pd.DataFrame([
    {
        "model": "Majority Class",
        "accuracy": majority_accuracy,
        "macro_f1": majority_macro_f1,
        "weighted_f1": majority_weighted_f1
    },
    {
        "model": "TF-IDF + Logistic Regression",
        "accuracy": accuracy_score(y_test, y_pred_lr),
        "macro_f1": f1_score(y_test, y_pred_lr, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred_lr, average="weighted")
    },
    {
        "model": "TF-IDF + Linear SVM",
        "accuracy": accuracy_score(y_test, y_pred_svm),
        "macro_f1": f1_score(y_test, y_pred_svm, average="macro"),
        "weighted_f1": f1_score(y_test, y_pred_svm, average="weighted")
    }
])

baseline_results

,model,accuracy,macro_f1,weighted_f1
0,Majority Class,0.26,0.027513,0.107302
1,TF-IDF + Logistic Regression,0.28,0.198070,0.291474
2,TF-IDF + Linear SVM,0.32,0.260000,0.329000


In [14]:
baseline_results.to_csv(
    "../evaluation/baseline_results.csv",
    index=False
)

print("Baseline results saved.")

Baseline results saved.


In [15]:
pd.read_csv("../evaluation/baseline_results.csv")

,model,accuracy,macro_f1,weighted_f1
0,Majority Class,0.26,0.027513,0.107302
1,TF-IDF + Logistic Regression,0.28,0.198070,0.291474
2,TF-IDF + Linear SVM,0.32,0.260000,0.329000


In [19]:
import os

print(os.getcwd())
print(os.listdir())

/Users/sam/Desktop/hiver_sde_assignment/notebooks
['.DS_Store', '01_dataset_audit.ipynb', 'evaluation', '02_baseline.ipynb', '.ipynb_checkpoints']


In [20]:
import os

for root, dirs, files in os.walk(".."):
    if "americanair_customer_pairs.csv" in files:
        print(os.path.join(root, "americanair_customer_pairs.csv"))

../data/processed/americanair_customer_pairs.csv


In [21]:
import pandas as pd

twcs = pd.read_csv("../data/twcs.csv")

print(twcs.columns.tolist())

['tweet_id', 'author_id', 'inbound', 'created_at', 'text', 'response_tweet_id', 'in_response_to_tweet_id']


In [22]:
aa_outbound = twcs[
    (twcs["inbound"] == False) &
    (twcs["author_id"] == "AmericanAir")
].copy()

print("AmericanAir outbound:", len(aa_outbound))

AmericanAir outbound: 36764


In [24]:
customer_tweets = twcs[twcs["inbound"] == True][
    ["tweet_id", "text"]
].copy()

historical_pairs = aa_outbound.merge(
    customer_tweets,
    left_on="in_response_to_tweet_id",
    right_on="tweet_id",
    how="inner",
    suffixes=("_response", "_customer")
)

historical_pairs = historical_pairs[
    ["tweet_id_customer", "text_customer", "text_response"]
].rename(columns={
    "tweet_id_customer": "customer_tweet_id",
    "text_customer": "customer_message",
    "text_response": "brand_response"
})

print("Historical customer-response pairs:", len(historical_pairs))
print(historical_pairs.head())

Historical customer-response pairs: 36531
   customer_tweet_id                                   customer_message  \
0                997  @AmericanAir Erica on the lax team is amazing ...   
1                999  @AmericanAir Could you have someone on your la...   
2               1002  Ben Tennyson and an American Airlines pilot. 🎃...   
3               1005  I’m sorry, what? It’s going to COST me $50 to ...   
4               1004  @AmericanAir Right, but I earned those. I also...   

                                      brand_response  
0  @115904 We'll be sure to pass along your kind ...  
1  @115904 Our apologies for the delay in respond...  
2  @115905 Aww, that's definitely a future pilot ...  
3  @115906 This is a great option for customers w...  
4          @115906 We're sorry for your frustration.  


In [26]:
import os

os.makedirs("../data/processed", exist_ok=True)

historical_pairs.to_csv(
    "../data/processed/americanair_customer_pairs.csv",
    index=False
)

print("Saved:", len(historical_pairs), "historical pairs")

Saved: 36531 historical pairs


In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retriever_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000
)

historical_vectors = retriever_vectorizer.fit_transform(
    historical_pairs["customer_message"]
)

print("Retriever ready:", historical_vectors.shape)

Retriever ready: (36531, 50000)


In [27]:
def retrieve_similar_messages(query, k=3):
    query_vector = retriever_vectorizer.transform([query])
    scores = cosine_similarity(query_vector, historical_vectors).flatten()

    top_indices = scores.argsort()[-k:][::-1]

    results = historical_pairs.iloc[top_indices].copy()
    results["similarity"] = scores[top_indices]

    return results[
        ["customer_message", "brand_response", "similarity"]
    ]

In [28]:
retrieve_similar_messages(
    "My flight was cancelled and I need another flight",
    k=3
)

,customer_message,brand_response,similarity
5392,@AmericanAir When will I know if my flight is ...,@224071 We're working very hard to find a full...,0.303461
32676,@AmericanAir my flight was cancelled and now b...,@757006 We avoid cancellations whenever possib...,0.300857
14298,"@AmericanAir hey! I'm a gold member, my flight...","@394240 We're sorry to hear that, Sara. If you...",0.277321


In [29]:
test_queries = [
    "My bag has been lost",
    "I need to change my flight",
    "I want a refund for my cancelled flight",
    "I cannot check in online",
    "Can I choose my seat?"
]

for query in test_queries:
    print("\nQUERY:", query)
    print(
        retrieve_similar_messages(query, k=2)[
            ["customer_message", "brand_response", "similarity"]
        ].to_string(index=False)
    )


QUERY: My bag has been lost
                                                                                                     customer_message                                                                                                                      brand_response  similarity
@AmericanAir My luggage has been lost, and the phone operators haven't been helpful at all. My bag tag is 4001003572. @746065 We're sorry this is taking longer than expected but we have more information to provide. We're sending you more info in DM.    0.402758
                             @AmericanAir Shame it doesn't do lost baggage reporting, my bag has been left in Miami 😕                                          @197352 We have a great Baggage tracker. You can take a peek here: https://t.co/vpkBeWiqTq    0.395515

QUERY: I need to change my flight
                                 customer_message                                                                             brand_response  similari

In [30]:
def retrieval_hit_rate(k=3):
    hits = 0

    for _, row in golden_annotation.iterrows():
        results = retrieve_similar_messages(
            row["customer_message"],
            k=k
        )

        retrieved_text = " ".join(
            results["customer_message"].tolist()
        ).lower()

        # Simple keyword-based sanity check
        intent_words = row["intent"].replace("_", " ").lower().split()

        if any(word in retrieved_text for word in intent_words):
            hits += 1

    return hits / len(golden_annotation)

print("Retrieval hit rate:", retrieval_hit_rate(k=3))

Retrieval hit rate: 0.265


In [31]:
def get_retrieval_context(query, k=3):
    results = retrieve_similar_messages(query, k=k)

    context = []

    for _, row in results.iterrows():
        context.append(
            f"Customer: {row['customer_message']}\n"
            f"AmericanAir response: {row['brand_response']}"
        )

    return "\n\n".join(context)

print(get_retrieval_context(
    "My flight was cancelled and I need help finding another flight"
))

Customer: @AmericanAir I need help with my flight.
AmericanAir response: @741022 Sure, how can we help?

Customer: @AmericanAir I need help
AmericanAir response: @628360 We'd be happy to help. What's going on?

Customer: @AmericanAir i need help!
AmericanAir response: @378328 We're here for you. What's going on, Lyssa?


In [32]:
def draft_reply(customer_message, k=3):
    context = get_retrieval_context(customer_message, k=k)

    return f"""Thanks for reaching out to American Airlines.

Based on similar customer cases, here's what we can suggest:

{context}

Our team can review your specific situation and help with the next steps."""
    
print(draft_reply(
    "My flight was cancelled and I need help finding another flight"
))

Thanks for reaching out to American Airlines.

Based on similar customer cases, here's what we can suggest:

Customer: @AmericanAir I need help with my flight.
AmericanAir response: @741022 Sure, how can we help?

Customer: @AmericanAir I need help
AmericanAir response: @628360 We'd be happy to help. What's going on?

Customer: @AmericanAir i need help!
AmericanAir response: @378328 We're here for you. What's going on, Lyssa?

Our team can review your specific situation and help with the next steps.


In [33]:
import os
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("OpenAI client ready")

OpenAI client ready


In [34]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input="Reply with exactly: API test successful."
)

print(response.output_text)

API test successful.


In [37]:
def generate_reply(customer_message, context):
    prompt = f"""
You are an American Airlines customer support agent.

Write a concise, helpful response to the customer.

Use the historical examples as evidence for how American Airlines handled similar cases.
Synthesize the examples rather than copying one response.
Only state information supported by the historical examples.
If the examples do not provide enough information to resolve the issue, say so and recommend contacting American Airlines support.

Customer message:
{customer_message}

Historical examples:
{context}

Return only the customer-facing reply.
"""

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    return response.output_text

In [38]:
test_message = "My flight was cancelled. What should I do?"

context = get_retrieval_context(test_message, k=3)

reply = generate_reply(test_message, context)

print(reply)

Please send us a DM with your record locator so we can take a closer look at your cancelled flight. If you booked through another company, please contact them directly for help as well.


In [39]:
def classify_intent(customer_message):
    return baseline_svm.predict([customer_message])[0]

In [40]:
test_message = "My flight was cancelled. What should I do?"

intent = classify_intent(test_message)

print("Predicted intent:", intent)

Predicted intent: follow_up


In [41]:
from sklearn.metrics import classification_report, f1_score

y_pred = baseline_svm.predict(X_test)

print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Weighted F1:", f1_score(y_test, y_pred, average="weighted"))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Macro F1: 0.26
Weighted F1: 0.32899999999999996

Classification Report:
                     precision    recall  f1-score   support

 airport_gate_staff       0.00      0.00      0.00         2
            baggage       0.00      0.00      0.00         3
     booking_change       0.17      0.50      0.25         2
           check_in       0.00      0.00      0.00         1
    contact_support       0.50      0.50      0.50         2
       fees_charges       0.20      0.33      0.25         3
  flight_disruption       0.29      0.40      0.33         5
 flight_information       0.00      0.00      0.00         2
          follow_up       0.50      0.50      0.50         8
    loyalty_upgrade       0.00      0.00      0.00         2
        non_support       0.57      0.31      0.40        13
   onboard_aircraft       0.00      0.00      0.00         1
      other_unclear       0.00      0.00      0.00         2
refund_compensation       1.00      1.00      1.00         2
            

In [42]:
baseline_results = pd.DataFrame({
    "model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Linear SVM"
    ],
    "macro_f1": [
        0.19807,
        0.26
    ],
    "weighted_f1": [
        0.291474,
        0.329
    ]
})

baseline_results.to_csv(
    "../evaluation/baseline_results.csv",
    index=False
)

baseline_results

,model,macro_f1,weighted_f1
0,TF-IDF + Logistic Regression,0.19807,0.291474
1,TF-IDF + Linear SVM,0.26000,0.329000


In [43]:
def classify_intent_llm(customer_message):
    intents = """
    flight_disruption
    baggage
    booking_change
    seat
    loyalty_upgrade
    fees_charges
    refund_compensation
    flight_information
    airport_gate_staff
    onboard_aircraft
    contact_support
    check_in
    follow_up
    non_support
    other_unclear
    """

    prompt = f"""
Classify the customer message into exactly one of these intents:

{intents}

Rules:
- flight_disruption: active cancellation, delay, missed connection, or other disruption
- flight_information: asking for flight/status information without an active problem
- baggage: baggage problems or baggage questions
- booking_change: changing, cancelling, or modifying a booking
- seat: seat assignment or seat selection
- loyalty_upgrade: upgrades, loyalty status, miles, or priority benefits
- fees_charges: fees, charges, or unexpected costs
- refund_compensation: requesting a refund, voucher, or compensation
- airport_gate_staff: airport, gate, or staff issues
- onboard_aircraft: issues with the aircraft or onboard experience
- contact_support: trying to reach/contact customer support
- check_in: check-in problems
- follow_up: following up on an existing support case
- non_support: not a customer support request
- other_unclear: unclear or insufficient information

Customer message:
{customer_message}

Return only the intent name.
"""

    response = client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )

    return response.output_text.strip()

In [44]:
test_message = "My flight was cancelled. What should I do?"

intent = classify_intent_llm(test_message)

print("LLM predicted intent:", intent)

LLM predicted intent: flight_disruption


In [46]:
llm_predictions = []

for message in X_test:
    prediction = classify_intent_llm(message)
    llm_predictions.append(prediction)

print("Finished:", len(llm_predictions))

Finished: 50


In [47]:
from sklearn.metrics import classification_report, f1_score

print("LLM Macro F1:", f1_score(y_test, llm_predictions, average="macro"))
print("LLM Weighted F1:", f1_score(y_test, llm_predictions, average="weighted"))

print("\nLLM Classification Report:")
print(classification_report(
    y_test,
    llm_predictions,
    zero_division=0
))

LLM Macro F1: 0.6536752136752138
LLM Weighted F1: 0.7104615384615385

LLM Classification Report:
                     precision    recall  f1-score   support

 airport_gate_staff       1.00      1.00      1.00         2
            baggage       0.67      0.67      0.67         3
     booking_change       0.00      0.00      0.00         2
           check_in       1.00      1.00      1.00         1
    contact_support       0.50      1.00      0.67         2
       fees_charges       0.67      0.67      0.67         3
  flight_disruption       0.80      0.80      0.80         5
 flight_information       0.50      0.50      0.50         2
          follow_up       0.80      0.50      0.62         8
    loyalty_upgrade       0.50      0.50      0.50         2
        non_support       0.92      0.92      0.92        13
   onboard_aircraft       0.50      1.00      0.67         1
      other_unclear       0.00      0.00      0.00         2
refund_compensation       0.67      1.00      0.

In [48]:
llm_results = pd.DataFrame({
    "model": [
        "TF-IDF + Logistic Regression",
        "TF-IDF + Linear SVM",
        "GPT-5.4-mini"
    ],
    "macro_f1": [
        0.19807,
        0.26,
        f1_score(y_test, llm_predictions, average="macro")
    ],
    "weighted_f1": [
        0.291474,
        0.329,
        f1_score(y_test, llm_predictions, average="weighted")
    ]
})

llm_results.to_csv(
    "../evaluation/model_results.csv",
    index=False
)

llm_results

,model,macro_f1,weighted_f1
0,TF-IDF + Logistic Regression,0.198070,0.291474
1,TF-IDF + Linear SVM,0.260000,0.329000
2,GPT-5.4-mini,0.653675,0.710462


In [49]:
def decide_escalation(intent, customer_message):
    escalation_intents = {
        "refund_compensation",
        "fees_charges",
        "flight_disruption",
        "baggage",
        "airport_gate_staff",
        "other_unclear"
    }

    if intent in escalation_intents:
        return {
            "decision": "escalate",
            "reason": f"{intent} may require case-specific handling."
        }

    return {
        "decision": "auto_handle",
        "reason": "The request can be answered using historical support guidance."
    }

In [50]:
decision = decide_escalation(
    "flight_disruption",
    "My flight was cancelled. What should I do?"
)

print(decision)

{'decision': 'escalate', 'reason': 'flight_disruption may require case-specific handling.'}


In [51]:
def run_agent(customer_message):
    # 1. Classify intent
    intent = classify_intent_llm(customer_message)

    # 2. Retrieve similar historical cases
    context = get_retrieval_context(customer_message, k=3)

    # 3. Generate grounded reply
    reply = generate_reply(customer_message, context)

    # 4. Decide whether to escalate
    decision = decide_escalation(intent, customer_message)

    return {
        "intent": intent,
        "reply": reply,
        "decision": decision["decision"],
        "reason": decision["reason"]
    }

In [52]:
result = run_agent(
    "My flight was cancelled. What should I do?"
)

print("Intent:", result["intent"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("Reply:", result["reply"])

Intent: flight_disruption
Decision: escalate
Reason: flight_disruption may require case-specific handling.
Reply: Please send us a DM with your record locator so we can take a closer look. If you booked through another company, please contact them as well.


In [53]:
test_messages = [
    "My bag never arrived at the airport.",
    "I need to change my flight to tomorrow.",
    "Why was I charged for my checked bag?",
    "Can I choose my seat?",
    "How do I check in online?"
]

for message in test_messages:
    result = run_agent(message)

    print("\nCustomer:", message)
    print("Intent:", result["intent"])
    print("Decision:", result["decision"])
    print("Reply:", result["reply"])


Customer: My bag never arrived at the airport.
Intent: baggage
Decision: escalate
Reply: Please have your bag inspected in person by our bag team at the airport. If you’d like us to take a closer look, follow and DM your record locator and the details of what happened.

Customer: I need to change my flight to tomorrow.
Intent: booking_change
Decision: auto_handle
Reply: We’d like to take a look at your reservation and help with the change. Please DM us your record locator and flight details, and our team can assist. If you prefer, you can also call 800-433-7300.

Customer: Why was I charged for my checked bag?
Intent: fees_charges
Decision: escalate
Reply: Please DM us your record locator and any additional details about the bag charge, and we’ll take a closer look. Based on previous cases, checked bag charges are generally nonrefundable.

Customer: Can I choose my seat?
Intent: seat
Decision: auto_handle
Reply: You’re welcome to select your seats anytime prior to check-in as well. If

In [57]:
import importlib
import src.agent as agent

importlib.reload(agent)

print("Updated agent module loaded")

Updated agent module loaded


In [58]:
results = agent.retrieve_similar_messages(
    "My flight was cancelled. What should I do?",
    historical_pairs,
    retriever_vectorizer,
    historical_vectors,
    k=3
)

results

,customer_tweet_id,customer_message,brand_response,similarity
14038,1147202,@AmericanAir did NOT recieve any email since y...,@390004 Please contact the company you've book...,0.387762
34708,2829210,@AmericanAir my flight out of Bali will possib...,@452840 If you send us a DM with your record l...,0.365995
23928,2037566,@AmericanAir I was unable to add a check bag o...,@602464 Checked bag fees are collected at the ...,0.340346


In [59]:
for _, row in results.iterrows():
    print("CUSTOMER:", row["customer_message"])
    print("RESPONSE:", row["brand_response"])
    print("SIMILARITY:", round(row["similarity"], 3))
    print("-" * 80)

CUSTOMER: @AmericanAir did NOT recieve any email since yesterday. What should I do? https://t.co/UwAnf296yF
RESPONSE: @390004 Please contact the company you've booked through if you're needing a new email.
SIMILARITY: 0.388
--------------------------------------------------------------------------------
CUSTOMER: @AmericanAir my flight out of Bali will possibly be delayed, what should i do about my connecting flights back to the US? @25053
RESPONSE: @452840 If you send us a DM with your record locator, we'd be happy to take a closer look.
SIMILARITY: 0.366
--------------------------------------------------------------------------------
CUSTOMER: @AmericanAir I was unable to add a check bag on app with check in. What should I do?
RESPONSE: @602464 Checked bag fees are collected at the airport, either at the kiosk or with one of our ticket counter agents.
SIMILARITY: 0.34
--------------------------------------------------------------------------------


In [60]:
test_message = "My flight was cancelled. What should I do?"

intent = classify_intent(test_message)

print("Intent:", intent)

Intent: flight_disruption


In [62]:
candidates = agent.retrieve_similar_messages(
    test_message,
    historical_pairs,
    retriever_vectorizer,
    historical_vectors,
    k=10
)

print("Candidates found:", len(candidates))

Candidates found: 6


In [63]:
selected = agent.select_relevant_examples(
    test_message,
    intent,
    candidates,
    k=3
)

print("Selected examples:", selected)

Selected examples: 36431, 34708, 14038


In [64]:
selected_ids = [int(x.strip()) for x in selected.split(",")]

candidates.loc[selected_ids]

,customer_tweet_id,customer_message,brand_response,similarity
36431,2976919,@AmericanAir If my flight does get canceled du...,@821008 We'll let everyone know of any changes...,0.325206
34708,2829210,@AmericanAir my flight out of Bali will possib...,@452840 If you send us a DM with your record l...,0.365995
14038,1147202,@AmericanAir did NOT recieve any email since y...,@390004 Please contact the company you've book...,0.387762


In [69]:
selected_rows = candidates.loc[selected_ids]

selected_context = "\n\n".join(
    f"Customer: {row['customer_message']}\n"
    f"AmericanAir response: {row['brand_response']}"
    for _, row in selected_rows.iterrows()
)

print(selected_context)

Customer: @AmericanAir If my flight does get canceled due to the recent events of pilots taking leave, what should I do in advance?
AmericanAir response: @821008 We'll let everyone know of any changes as early as possible. If your flight is cancelled you can call us to get rebooked.

Customer: Hey @AmericanAir my flight was cancelled and I was rebooked with a different company that isn’t checking people in for hours. What do I do?
AmericanAir response: @431681 We'd be happy to take a closer look at your reservation. Please follow and slide into DMs with your record locator.

Customer: @AmericanAir And now the flight was cancelled... again... because of maintenance.
AmericanAir response: @439232 Safety for our customers and crew is of the utmost importance. Please let us know if you need help to rebook.


In [70]:
reply = generate_reply(
    test_message,
    selected_context
)

print(reply)

I’m sorry your flight was cancelled. If you need help rebooking, please contact us so we can take a closer look at your reservation and get you the next available option. If you already have your record locator, have it ready when you reach out.


In [72]:
result = agent.run_agent(
    "My flight was cancelled. What should I do?",
    historical_pairs,
    retriever_vectorizer,
    historical_vectors
)

print("Intent:", result["intent"])
print("Decision:", result["decision"])
print("Reason:", result["reason"])
print("Reply:", result["reply"])

Intent: flight_disruption
Decision: escalate
Reason: flight_disruption may require case-specific handling.
Reply: I’m sorry about the cancellation. If your flight was cancelled, you can call us to get rebooked, and we’d be happy to take a closer look at your reservation if you share your record locator.
